In [ ]:

import os
import tqdm
from datasets import load_dataset
from google.colab import userdata, files

# --- CONFIGURATION ---
OUTPUT_FILE = "vital_slm_150m_final.txt"

# 📉 AGGRESSIVE LIMITS (Based on your 570M result)
LIMIT_PUBMED_ABSTRACTS = 20000   # Cut down to 20k to prevent overshoot
LIMIT_FINEWEB_EDU      = 25000   # Good balance for grammar

def clean_text(text):
    if not text: return ""
    return text.strip().replace("\r", " ").replace("\n\n", "\n")

def save_stream_to_file(f, dataset_iter, limit, source_name, text_formatter):
    print(f" -> ⏳ Processing {source_name} (Target: {limit})...")
    count = 0
    pbar = tqdm.tqdm(total=limit, desc=source_name, unit="doc")

    while count < limit:
        try:
            row = next(dataset_iter)
            content = text_formatter(row)
            if content and len(content) > 50:
                f.write(content + "\n<|endoftext|>\n")
                count += 1
                pbar.update(1)
        except StopIteration:
            print(f" -> ⚠️ Source {source_name} ran out of data.")
            break
        except Exception:
            continue
    print(f" -> ✅ Saved {count} items from {source_name}.\n")

def save_full_dataset(f, dataset, source_name, text_formatter):
    print(f" -> ⏳ Processing {source_name} (Full Dataset)...")
    count = 0
    for row in tqdm.tqdm(dataset, desc=source_name, unit="doc"):
        try:
            content = text_formatter(row)
            if content and len(content) > 30:
                f.write(content + "\n<|endoftext|>\n")
                count += 1
        except:
            continue
    print(f" -> ✅ Saved {count} items from {source_name}.\n")

# --- MAIN EXECUTION ---
if os.path.exists(OUTPUT_FILE):
    os.remove(OUTPUT_FILE)

print(f"🚀 Starting Precision Data Generation (Target: ~150M Tokens)...")
with open(OUTPUT_FILE, "a", encoding="utf-8") as f:

    # 1. PubMed (Restricted to 20k)
    print("[1/6] Downloading PubMed Abstracts (Limit: 20k)...")
    try:
        ds_pubmed = load_dataset("ccdv/pubmed-summarization", split="train", streaming=True)
        save_stream_to_file(f, iter(ds_pubmed), LIMIT_PUBMED_ABSTRACTS, "PubMed",
                            lambda r: f"Abstract: {clean_text(r['article'])}")
    except Exception as e: print(f"⚠️ Error with PubMed: {e}")

    # 2. ChatDoctor (Full)
    print("[2/6] Downloading ChatDoctor (Full)...")
    try:
        ds_chat = load_dataset("lavita/ChatDoctor-HealthCareMagic-100k", split="train")
        save_full_dataset(f, ds_chat, "ChatDoctor",
                          lambda r: f"Patient: {clean_text(r['input'])}\nDoctor: {clean_text(r['output'])}")
    except Exception as e: print(f"⚠️ Error with ChatDoctor: {e}")

    # 3. WikiDoc (Full)
    print("[3/6] Downloading Medical WikiDoc (Full)...")
    try:
        ds_wiki = load_dataset("medalpaca/medical_meadow_wikidoc", split="train")
        save_full_dataset(f, ds_wiki, "WikiDoc",
                          lambda r: f"Term: {clean_text(r['input'])}\nDefinition: {clean_text(r['output'])}")
    except Exception as e: print(f"⚠️ Error with WikiDoc: {e}")

    # 4. PubMedQA (Full)
    print("[4/6] Downloading PubMedQA (Full)...")
    try:
        ds_pqa = load_dataset("pubmed_qa", "pqa_labeled", split="train")
        save_full_dataset(f, ds_pqa, "PubMedQA",
                          lambda r: f"Question: {clean_text(r['question'])}\nAnswer: {clean_text(r['long_answer'])}")
    except Exception as e: print(f"⚠️ Error with PubMedQA: {e}")

    # 5. MedQA (Full)
    print("[5/6] Downloading MedQA (Full)...")
    try:
        ds_medqa = load_dataset("medalpaca/medical_meadow_medqa", split="train")
        save_full_dataset(f, ds_medqa, "MedQA",
                          lambda r: f"Question: {clean_text(r['input'])}\nAnswer: {clean_text(r['output'])}")
    except Exception as e: print(f"⚠️ Error with MedQA: {e}")

    # 6. FineWeb (Restricted to 25k)
    print("[6/6] Downloading FineWeb-Edu (Limit: 25k)...")
    try:
        ds_fw = load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT", split="train", streaming=True)
        save_stream_to_file(f, iter(ds_fw), LIMIT_FINEWEB_EDU, "FineWeb-Edu",
                            lambda r: clean_text(r['text']))
    except Exception as e: print(f"⚠️ Error with FineWeb: {e}")

# --- VERIFICATION ---
if os.path.exists(OUTPUT_FILE):
    file_size = os.path.getsize(OUTPUT_FILE)
    size_mb = file_size / (1024 * 1024)
    est_tokens = file_size / 3.5 / 1_000_000

    print(f"\n✅ GENERATION COMPLETE!")
    print(f"📂 File: {OUTPUT_FILE}")
    print(f"📊 Size: {size_mb:.2f} MB")
    print(f"🔢 Est. Tokens: {est_tokens:.2f} Million")



In [ ]:

from tokenizers import ByteLevelBPETokenizer
from tokenizers.processors import BertProcessing
import os

# --- CONFIGURATION ---
DATA_FILE = "vital_slm_150m_final.txt"  # The file you are currently generating
VOCAB_SIZE = 16384  
OUTPUT_DIR = "vital_tokenizer"

print(f"🚀 Training Tokenizer on {DATA_FILE}...")
print(f"   Target Vocab Size: {VOCAB_SIZE}")

# 2. Initialize Tokenizer
tokenizer = ByteLevelBPETokenizer()

# 3. Train it!
# special_tokens: Critical for the model to know start/end/padding
tokenizer.train(files=[DATA_FILE], vocab_size=VOCAB_SIZE, min_frequency=2, special_tokens=[
    "<|endoftext|>",  # 0: End of text / Separator
    "<|padding|>",    # 1: Padding
    "<|unknown|>",    # 2: Unknown words
    "<|mask|>",       # 3: Masking (optional)
])

# 4. Save it
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

tokenizer.save_model(OUTPUT_DIR)

print(f"✅ Tokenizer saved to folder: '{OUTPUT_DIR}'")
print("   Files generated: vocab.json, merges.txt")

# --- TEST IT ---
# Let's see how it handles a complex medical word
sample_text = "Patient has acute myocardial infarction."
encoded = tokenizer.encode(sample_text)
print(f"\n🔍 Test Encoding:")
print(f"   Input: '{sample_text}'")
print(f"   Tokens: {encoded.tokens}")
print(f"   IDs:    {encoded.ids}")

In [ ]:
import os
import numpy as np
import random
from tokenizers import ByteLevelBPETokenizer
from tqdm import tqdm

# --- CONFIGURATION ---
INPUT_FILE = "vital_slm_150m_final.txt"  
TOKENIZER_DIR = "vital_tokenizer"
TRAIN_FILE = "train.bin"
VAL_FILE = "val.bin"
TRAIN_RATIO = 0.9

# 1. Load Tokenizer
print(f"📂 Loading Tokenizer from '{TOKENIZER_DIR}'...")
try:
    tokenizer = ByteLevelBPETokenizer(
        f"{TOKENIZER_DIR}/vocab.json",
        f"{TOKENIZER_DIR}/merges.txt"
    )
except Exception as e:
    print(f"❌ Error: {e}")
    exit()

# 2. Read & SHUFFLE (Critical Step!)
print(f"📊 Reading lines from {INPUT_FILE}...")
with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    lines = f.readlines()

print(f"🎲 Shuffling {len(lines):,} lines to ensure balanced training...")
random.shuffle(lines) # <--- THIS FIXES THE DATA DISTRIBUTION

total_lines = len(lines)
split_idx = int(total_lines * TRAIN_RATIO)

print(f"   Split:       {split_idx:,} (Train) | {total_lines - split_idx:,} (Val)")

# 3. Stream Processing (Text -> Binary)
print("🚀 Tokenizing and writing to binary files...")

# Create empty files
open(TRAIN_FILE, 'w').close()
open(VAL_FILE, 'w').close()

with open(TRAIN_FILE, 'wb') as f_train, open(VAL_FILE, 'wb') as f_val:

    for i, line in enumerate(tqdm(lines)):
        # Skip empty lines
        if not line.strip(): continue

        # Encode
        # Note: Our text file already has <|endoftext|>, so the tokenizer
        # will handle it naturally as a token ID.
        encoded = tokenizer.encode(line)

        # Optimization: uint16
        # Vocab is 16384, which fits in uint16 (max 65535).
        # This reduces file size by 75% compared to int64.
        ids = np.array(encoded.ids, dtype=np.uint16)

        # Write raw bytes
        if i < split_idx:
            f_train.write(ids.tobytes())
        else:
            f_val.write(ids.tobytes())

print(f"\n✅ Success! Binary datasets created.")
print(f"   - {TRAIN_FILE}")
print(f"   - {VAL_FILE}")

In [ ]:
from tokenizers import ByteLevelBPETokenizer

# Define the folder path containing vocab.json and merges.txt
vocab_path = "/kaggle/input/vitallm-updated-tokenizer/updated_vocab.json"
merges_path = "/kaggle/input/vitallm-updated-tokenizer/updated_merges.txt"

# Initialize directly
tokenizer = ByteLevelBPETokenizer(
    vocab=vocab_path,
    merges=merges_path
)

print("✅ Tokenizer loaded successfully!")

In [ ]:
import torch
import numpy as np
import os

# --- CONFIGURATION ---
batch_size = 64
block_size = 256
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# --- 1. LOAD DATA (Memmap Mode) ---
# We point to the local files you just generated ('train.bin' and 'val.bin')
# If you are restarting a Kaggle session, these might be in /kaggle/working/
train_path = '/kaggle/input/vitallm-updated-data/train_v1.bin'
val_path = '/kaggle/input/vitallm-updated-data/val_v1.bin'

print(f"📂 Loading memmaps from {train_path} and {val_path}...")

# Check if files exist to avoid crashing later
if not os.path.exists(train_path) or not os.path.exists(val_path):
    print("❌ Error: Binary files not found! Did you run the 'prepare_bin_files.py' script?")
else:
    # Load memory-mapped files (Instant access, low RAM usage)
    train_data = np.memmap(train_path, dtype=np.uint16, mode='r')
    val_data = np.memmap(val_path, dtype=np.uint16, mode='r')
    print(f"✅ Loaded! Train tokens: {len(train_data):,}, Val tokens: {len(val_data):,}")

# --- 2. GET_BATCH FUNCTION ---
def get_batch(split):
    # Select the correct dataset
    data = train_data if split == 'train' else val_data

    # Generate random starting indices
    # We subtract block_size to ensure we don't read off the end of the file
    ix = torch.randint(len(data) - block_size, (batch_size,))

    # Stack inputs (x) and targets (y)
    # 1. Slice the memmap (fast disk access)
    # 2. Convert to numpy int64 (Required for PyTorch Embeddings)
    # 3. Convert to Torch Tensor
    x = torch.stack([torch.from_numpy((data[i:i+block_size]).astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy((data[i+1:i+1+block_size]).astype(np.int64)) for i in ix])

    # Move to GPU (Asynchronous is faster)
    if device == 'cuda':
        # non_blocking=True allows the GPU to compute while data is still moving
        x = x.pin_memory().to(device, non_blocking=True)
        y = y.pin_memory().to(device, non_blocking=True)
    else:
        x, y = x.to(device), y.to(device)

    return x, y

Model Architecture

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from dataclasses import dataclass
import numpy as np
from tqdm import tqdm
from contextlib import nullcontext
import os

class LayerNorm(nn.Module):
    def __init__(self, ndim, bias=True, eps=1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None
        self.eps = eps

    def forward(self, x):
        return F.layer_norm(x, x.shape[-1:], self.weight, self.bias, self.eps)


class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.flash = hasattr(F, 'scaled_dot_product_attention')
        if not self.flash:
            self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                        .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        if self.flash:
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=self.attn_dropout.p if self.training else 0.0, is_causal=True)
        else:
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
            att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            y = att @ v

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.c_proj(y))
        return y


class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        # SwiGLU typically keeps the hidden dimension at 4 * n_embd (like LLaMA), 
        # or uses 8/3 * n_embd to maintain the same parameter count as a standard MLP.
        # Here we stick to 4 * n_embd for maximum capacity.
        hidden_dim = 4 * config.n_embd
        
        # w1: Gate Projection
        self.w1 = nn.Linear(config.n_embd, hidden_dim, bias=config.bias)
        # w2: Value Projection
        self.w2 = nn.Linear(config.n_embd, hidden_dim, bias=config.bias)
        # c_proj: Output Projection (Down projection)
        self.c_proj = nn.Linear(hidden_dim, config.n_embd, bias=config.bias)
        
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        # SwiGLU Logic: (SiLU(Gate) * Value) -> Projection
        # 1. Gate path: w1(x) -> SiLU
        # 2. Value path: w2(x)
        # 3. Element-wise multiply
        x = F.silu(self.w1(x)) * self.w2(x)
        
        # 4. Output projection
        return self.dropout(self.c_proj(x))


class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln1 = LayerNorm(config.n_embd, config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln2 = LayerNorm(config.n_embd, config.bias)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

@dataclass
class SLMConfig:
    block_size: int
    vocab_size: int
    n_layer: int
    n_head: int
    n_embd: int
    dropout: float = 0.0
    bias: bool = True

class SLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte=nn.Embedding(config.vocab_size, config.n_embd),
            wpe=nn.Embedding(config.block_size, config.n_embd),
            drop=nn.Dropout(config.dropout),
            h=nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f=LayerNorm(config.n_embd, config.bias),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight  # weight tying

        self.apply(self._init_weights)
        # Apply special scaled init to the residual projections, c_proj
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        device = idx.device
        b, t = idx.size()
        assert t <= self.config.block_size
        pos = torch.arange(0, t, dtype=torch.long, device=device)

        tok_emb = self.transformer.wte(idx)
        pos_emb = self.transformer.wpe(pos)
        x = self.transformer.drop(tok_emb + pos_emb)
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)

        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
            return logits, loss
        else:
            logits = self.lm_head(x[:, [-1], :])
            return logits, None

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """
        Generate tokens given a conditioning sequence.
        idx: Tensor of shape (B, T)
        """
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

In [ ]:
config = SLMConfig(
    vocab_size=16384,   # Updated: Matches your new tokenizer (vocab.json)
    block_size=256,     # Context window
    n_embd=320,         # Updated: Dimension size for ~25M-30M params
    n_head=8,           # Updated: 320 / 8 = 40 dimensions per head
    n_layer=12,         # Updated: Deeper network (was 12 in the snippet)
    dropout=0.1,
    bias=True           # Kept True as per your class definition
)

print(f"🚀 Initializing Model with config: {config}")
model = SLM(config)

# Verify Parameter Count
n_params = sum(p.numel() for p in model.parameters())
print(f"🧠 Total Parameters: {n_params/1e6:.2f} Million")

In [ ]:
# --- UPDATED ESTIMATE LOSS FUNCTION ---
import torch
@torch.no_grad()  # This decorator replaces "with torch.inference_mode():"
def estimate_loss():  # <--- REMOVED 'model' ARGUMENT
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            with ctx:
                logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [ ]:
# Training Config
import torch
from contextlib import nullcontext

learning_rate = 6e-4 #more stable training, earlier 1e-4
max_iters = 19000 #increase from 25000
warmup_steps = 1000 #smoother initial train, earlier 100
min_lr = 1e-5 #lower rate, earlier 5e-4
eval_iters = 500
batch_size = 32
block_size = 256

gradient_accumulation_steps = 4 # reduced from 50

device =  "cuda" if torch.cuda.is_available() else "cpu"
device_type = 'cuda' if 'cuda' in device else 'cpu' # for later use in torch.autocast


#dtype = 'bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'float16' # 'float32', 'bfloat16', or 'float16', the latter will auto implement a GradScaler
dtype = 'bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'float16' # 'float32', 'bfloat16', or 'float16', the latter will auto implement a GradScaler
ptdtype = {'float32': torch.float32, 'bfloat16': torch.bfloat16, 'float16': torch.float16}[dtype]

ctx = nullcontext() if device_type == 'cpu' else torch.amp.autocast(device_type=device_type, dtype=ptdtype)

torch.set_default_device(device)
torch.manual_seed(42)

In [ ]:
import time
import math
from contextlib import nullcontext

# --- 1. HYPERPARAMETERS (The "1.1 Epoch" Strategy) ---
max_iters = 22000        # Optimized for ~140M tokens
warmup_steps = 1000      # 1k steps to warm up
learning_rate = 5e-4     # Max LR
min_lr = 5e-5            # Min LR (10% of max)
eval_interval = 500      # Check validation every 500 steps
eval_iters = 200         # Smooth validation loss over 200 batches
weight_decay = 0.1       # Standard regularization

# --- 2. SETUP MIXED PRECISION (Crucial for P100 Speed) ---
# P100 supports float16 nicely.
dtype = 'float16' 
device_type = 'cuda' if 'cuda' in device else 'cpu'
ptdtype = {'float32': torch.float32, 'bfloat16': torch.bfloat16, 'float16': torch.float16}[dtype]
ctx = nullcontext() if device_type == 'cpu' else torch.amp.autocast(device_type=device_type, dtype=ptdtype)
scaler = torch.cuda.amp.GradScaler(enabled=(dtype == 'float16'))

# --- 3. OPTIMIZER (With Weight Decay Fix) ---
# Separate parameters into those that decay (weights) and those that don't (biases, layernorms)
param_dict = {pn: p for pn, p in model.named_parameters() if p.requires_grad}
decay_params = [p for n, p in param_dict.items() if p.dim() >= 2]
nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]

optim_groups = [
    {'params': decay_params, 'weight_decay': weight_decay},
    {'params': nodecay_params, 'weight_decay': 0.0}
]

optimizer = torch.optim.AdamW(optim_groups, lr=learning_rate, betas=(0.9, 0.95), eps=1e-9)

# --- 4. SCHEDULER (Cosine Decay with Warmup) ---
# This ensures the LR goes up smoothly then decays down to min_lr at 19000 steps
def get_lr(it):
    # 1) Linear warmup
    if it < warmup_steps:
        return learning_rate * (it + 1) / (warmup_steps + 1)
    # 2) If it > max_iters, return min learning rate
    if it > max_iters:
        return min_lr
    # 3) Cosine decay down to min learning rate
    decay_ratio = (it - warmup_steps) / (max_iters - warmup_steps)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (learning_rate - min_lr)

# --- 5. INITIALIZATION ---
model.to(device)
best_val_loss = float('inf')
train_loss_list = []
val_loss_list = []
lr_list = []

print(f"🚀 Starting training for {max_iters} iterations on {device}...")
print(f"   Model: ~25M Params | Batch: 64 | Context: 256")

t0 = time.time()

for iter in range(max_iters):

    # --- A. SET LEARNING RATE ---
    lr = get_lr(iter)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr
    lr_list.append(lr)

    # --- B. EVALUATION ---
    if iter % eval_interval == 0 and iter > 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}, lr {lr:.2e}")
        
        train_loss_list.append(losses['train'].item())
        val_loss_list.append(losses['val'].item())

        # Save Best Model
        if losses['val'] < best_val_loss:
            best_val_loss = losses['val']
            torch.save(model.state_dict(), "vital_lm_25m_swiglu_best.pt")
            print(f"   -> 💾 Saved Best Model (Val Loss: {best_val_loss:.4f})")

    # --- C. TRAINING STEP ---
    # 1. Get Batch
    xb, yb = get_batch('train')

    # 2. Forward Pass (Mixed Precision)
    with ctx:
        logits, loss = model(xb, yb)

    # 3. Backward Pass
    # Scaling is needed because float16 gradients can be tiny
    scaler.scale(loss).backward()
    
    # 4. Gradient Clipping (Unscale first!)
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

    # 5. Optimizer Step
    scaler.step(optimizer)
    scaler.update()
    
    # 6. Flush Gradients (set_to_none is faster than zero_grad)
    optimizer.zero_grad(set_to_none=True)

    # --- D. LOGGING SPEED ---
    if iter % 100 == 0:
        t1 = time.time()
        dt = t1 - t0
        t0 = t1
        # Calculate tokens per second
        tps = (batch_size * config.block_size * 100) / dt
        print(f"iter {iter} | loss {loss.item():.4f} | time {dt*1000:.2f}ms | speed {tps:.0f} tok/s")

# --- FINAL SAVE ---
print("\n✅ Training Complete.")
torch.save(model.state_dict(), "vital_lm_25m_swiglu_final.pt")
print("💾 Final model saved as 'vital_lm_25m_swiglu_final.pt'")

In [ ]:
import matplotlib.pyplot as plt
import torch
import numpy as np

# --- 1. DATA PREPARATION ---

# Helper to smooth jittery lines
def smooth(scalars, weight=0.85):
    if len(scalars) == 0: return []
    last = scalars[0]
    smoothed = list()
    for point in scalars:
        smoothed_val = last * weight + (1 - weight) * point
        smoothed.append(smoothed_val)
        last = smoothed_val
    return smoothed

# Safe conversion to CPU list
def to_list(data):
    if isinstance(data, list):
        if len(data) == 0: return []
        if isinstance(data[0], torch.Tensor):
            return [x.cpu().detach().item() for x in data]
        return data
    return data

# Load Data
t_loss = to_list(train_loss_list)
v_loss = to_list(validation_loss_list)
lrs = to_list(lr_list)

# Check if grad_norm_list exists (it might not in the optimized loop)
try:
    grads = to_list(grad_norm_list)
except NameError:
    grads = []

# X-Axis alignment
# Since losses are saved every 'eval_interval' (e.g., 500 steps), we need correct X-steps
eval_interval = 500  # Adjust this if you changed your config!
eval_steps = np.arange(0, len(t_loss) * eval_interval, eval_interval)
total_steps = np.arange(0, len(lrs))

# --- GRAPH 1: LOSS CURVE (The Standard View) ---
plt.figure(figsize=(10, 6))
plt.plot(eval_steps, t_loss, 'g-', alpha=0.3, label='Raw Train Loss')
plt.plot(eval_steps, smooth(t_loss), 'g-', linewidth=2.5, label='Smoothed Train Loss')
plt.plot(eval_steps, v_loss, 'r--', linewidth=2.5, label='Validation Loss')
plt.title(f'1. Main Training Curve (Min Val Loss: {min(v_loss):.4f})', fontsize=14)
plt.xlabel('Training Steps')
plt.ylabel('Cross Entropy Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# --- GRAPH 2: PERPLEXITY (The "Intelligence" Score) ---
# PPL = exp(loss). It measures how "surprised" the model is. Lower is better.
val_ppl = [np.exp(min(l, 10)) for l in v_loss]  # Cap at 10 to prevent explosion
plt.figure(figsize=(10, 6))
plt.plot(eval_steps, val_ppl, 'orange', linewidth=2.5, marker='o', markersize=4)
plt.title('2. Perplexity (Fluency Check)', fontsize=14)
plt.xlabel('Training Steps')
plt.ylabel('Perplexity (Lower = Better)')
plt.grid(True, alpha=0.3)
plt.show()

# --- GRAPH 3: GENERALIZATION GAP (New!) ---
# Measures: Validation Loss - Training Loss
# > 0.0 : Normal (Validation is harder than Train)
# Increasing Trend : OVERFITTING WARNING
gap = np.array(v_loss) - np.array(t_loss)
plt.figure(figsize=(10, 6))
plt.fill_between(eval_steps, gap, 0, where=(gap>0), color='red', alpha=0.3, label='Overfitting Risk')
plt.fill_between(eval_steps, gap, 0, where=(gap<=0), color='blue', alpha=0.3, label='Underfitting / Good Gen')
plt.plot(eval_steps, gap, 'k-', linewidth=2)
plt.title('3. Generalization Gap (Keep this Low!)', fontsize=14)
plt.xlabel('Training Steps')
plt.ylabel('Val Loss - Train Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# --- GRAPH 4: LEARNING VELOCITY (New!) ---
# Measures: How fast is the loss dropping? (Derivative of loss)
# Near 0 = Convergence (Model stopped learning)
if len(t_loss) > 1:
    velocity = np.diff(smooth(t_loss))
    plt.figure(figsize=(10, 6))
    plt.plot(eval_steps[1:], velocity, 'purple', linewidth=2)
    plt.axhline(0, color='black', linestyle='--')
    plt.title('4. Learning Velocity (Speed of Improvement)', fontsize=14)
    plt.xlabel('Training Steps')
    plt.ylabel('Loss Change per Interval')
    plt.grid(True, alpha=0.3)
    plt.show()

# --- GRAPH 5: LEARNING RATE ---
if len(lrs) > 0:
    plt.figure(figsize=(10, 6))
    plt.plot(total_steps, lrs, 'b-', linewidth=2)
    plt.title('5. Learning Rate Schedule', fontsize=14)
    plt.xlabel('Steps')
    plt.ylabel('Learning Rate')
    plt.grid(True, alpha=0.3)
    plt.show()

# --- GRAPH 6: GRADIENT NORM (Optional) ---
if len(grads) > 0:
    plt.figure(figsize=(10, 6))
    plt.plot(grads, 'gray', alpha=0.4, label='Raw')
    plt.plot(smooth(grads, 0.95), 'k-', linewidth=2, label='Smoothed')
    plt.title('6. Gradient Norm (Stability Check)', fontsize=14)
    plt.xlabel('Steps')
    plt.ylabel('Norm Magnitude')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

In [ ]:
# --- CORRECTED INFERENCE CELL ---
import torch
from tokenizers import ByteLevelBPETokenizer

# 1. Define the EXACT Config used in training
# (Must match what you defined in Cell 4)
config = SLMConfig(
    vocab_size=16384,
    block_size=256,
    n_embd=320,
    n_head=8,
    n_layer=12,
    dropout=0.1,
    bias=True
)

model = SLM(config).to(device)

# 2. Load Model Weights
# Try 'best' first, then 'final'
MODEL_PATH = "/kaggle/working/vital_lm_25m_swiglu_best.pt" 

print(f"📂 Loading weights from {MODEL_PATH}...")
try:
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    print("✅ Model weights loaded!")
except FileNotFoundError:
    print(f"⚠️ 'best.pt' not found. Trying final model...")
    try:
        MODEL_PATH = "/kaggle/working/vital_lm_25m_swiglu_final.pt"
        model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
        print("✅ Final model weights loaded!")
    except:
        print("❌ CRITICAL: No model file found. Training might have failed.")
        # Stop here if no model
        raise

model.eval()

# 3. Load Tokenizer (Updated Filenames)
# Note: Matching the filenames you used in Cell 1
TOKENIZER_DIR = "/kaggle/input/vitallm-updated-tokenizer" 
vocab_file = f"{TOKENIZER_DIR}/updated_vocab.json"   # <--- FIXED NAME
merges_file = f"{TOKENIZER_DIR}/updated_merges.txt"  # <--- FIXED NAME

print(f"📂 Loading Tokenizer...")
try:
    tokenizer = ByteLevelBPETokenizer(vocab_file, merges_file)
    print("✅ Tokenizer loaded!")
except Exception as e:
    print(f"❌ Tokenizer Error: {e}")
    print(f"Checking path: {vocab_file}")

# 4. Chat Function (Strict settings for 25M model)
def chat(prompt, max_tokens=64, temp=0.3): # Low temp is key for small models!
    ids = tokenizer.encode(prompt).ids
    idx = torch.tensor(ids, dtype=torch.long).unsqueeze(0).to(device)
    
    print(f"\nUser: {prompt}")
    print("VitalLM: ", end="", flush=True)
    
    # Generate
    generated_idx = model.generate(idx, max_new_tokens=max_tokens, temperature=temp)
    
    # Decode
    full_text = tokenizer.decode(generated_idx[0].tolist())
    
    # Show only the new part (simple strip)
    answer = full_text[len(prompt):]
    print(answer)
    print("-" * 40)

# 5. Test It
chat("Patient: I have a severe headache and fever. Doctor:")
chat("Question: What is the main cause of malaria? Answer:")

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from dataclasses import dataclass
from tokenizers import ByteLevelBPETokenizer

# --- 1. DEFINE THE ARCHITECTURE (Must match training exactly) ---
@dataclass
class SLMConfig:
    block_size: int = 256
    vocab_size: int = 16384
    n_layer: int = 12
    n_head: int = 8
    n_embd: int = 320
    dropout: float = 0.0
    bias: bool = True

class LayerNorm(nn.Module):
    def __init__(self, ndim, bias=True, eps=1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None
        self.eps = eps
    def forward(self, x):
        return F.layer_norm(x, x.shape[-1:], self.weight, self.bias, self.eps)

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        
        # --- THE FIX IS HERE: Restore the Flash Attention Check ---
        self.flash = hasattr(F, 'scaled_dot_product_attention')
        if not self.flash:
            self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                        .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        # --- RESTORE FORWARD LOGIC ---
        if self.flash:
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=0.0, is_causal=True)
        else:
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
            att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))
            att = F.softmax(att, dim=-1)
            y = att @ v

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y))

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        hidden_dim = 4 * config.n_embd
        self.w1 = nn.Linear(config.n_embd, hidden_dim, bias=config.bias)
        self.w2 = nn.Linear(config.n_embd, hidden_dim, bias=config.bias)
        self.c_proj = nn.Linear(hidden_dim, config.n_embd, bias=config.bias)
    def forward(self, x):
        x = F.silu(self.w1(x)) * self.w2(x)
        return self.c_proj(x)

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln1 = LayerNorm(config.n_embd, config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln2 = LayerNorm(config.n_embd, config.bias)
        self.mlp = MLP(config)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class SLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte=nn.Embedding(config.vocab_size, config.n_embd),
            wpe=nn.Embedding(config.block_size, config.n_embd),
            h=nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f=LayerNorm(config.n_embd, config.bias),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight 

    def forward(self, idx):
        b, t = idx.size()
        pos = torch.arange(0, t, dtype=torch.long, device=idx.device)
        tok_emb = self.transformer.wte(idx)
        pos_emb = self.transformer.wpe(pos)
        x = tok_emb + pos_emb
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x[:, [-1], :])
        return logits

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
            logits = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

# --- 2. LOAD MODEL & TOKENIZER ---
device = 'cuda' if torch.cuda.is_available() else 'cpu'
config = SLMConfig()
model = SLM(config).to(device)

# UPDATE THESE PATHS to where you uploaded your files
model_path = "/kaggle/input/vital-25m-model/pytorch/default/1/vital_lm_25m_swiglu_best.pt"  
tokenizer_path = "/kaggle/input/vitallm-updated-tokenizer" # Folder containing vocab.json and merges.txt

print(f"📂 Loading weights from {model_path}...")
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()
print("✅ Model loaded!")

print("📂 Loading Tokenizer...")
tokenizer = ByteLevelBPETokenizer(f"{tokenizer_path}/updated_vocab.json", f"{tokenizer_path}/updated_merges.txt")

# --- 3. CHAT FUNCTION ---
def chat(prompt):
    ids = tokenizer.encode(prompt).ids
    idx = torch.tensor(ids, dtype=torch.long).unsqueeze(0).to(device)
    print(f"\nUser: {prompt}\nVitalLM:", end=" ")
    
    # Strict settings for best quality
    out = model.generate(idx, max_new_tokens=120, temperature=0.3, top_k=40)
    
    decoded = tokenizer.decode(out[0].tolist())
    print(decoded[len(prompt):])
    print("-" * 30)

# --- 4. TEST ---
chat("Patient: I have a severe headache and sensitive to light. Doctor:")
chat("Question: What is the treatment for malaria? Answer:")

📂 Loading weights from /kaggle/input/vital-25m-model/pytorch/default/1/vital_lm_25m_swiglu_best.pt...
✅ Model loaded!
📂 Loading Tokenizer...

User: Patient: I have a severe headache and sensitive to light. Doctor:
VitalLM:  I have a very bad headache and have been taking a deep breath. I have a little bit of pain in my chest, and it has been going on for a few weeks. I have been to the doctor and have been to the doctor and they have said I have a sinus infection. I have a few questions. I have a very bad headache and have had a cold for a few weeks now. I am a 20 year old female. I have been to the doctor and they have said I have a sinus infection. I have had a sinus infection and have had a sinus infection. I have
------------------------------

User: Question: What is the treatment for malaria? Answer:
VitalLM: 
 the overall survival rate of patients with stage iv disease was significantly higher than that of patients with stage iv disease . 
 the mean age of the patients was 18.2 

In [6]:
# --- 10 TEST CASES FOR VITAL-LM ---

test_prompts = [
    # Category 1: Symptom Analysis
    "Patient: I have a running nose, sneezing, and itchy eyes. Doctor:",
    "Patient: I feel a burning sensation in my chest after eating spicy food. Doctor:",
    "Patient: My ankle is swollen and painful after I twisted it while running. Doctor:",
    "Patient: I have a high fever, body aches, and chills. Doctor:",

    # Category 2: Drug & Treatment Knowledge
    "Question: What is the primary use of Insulin? Answer:",
    "Question: What is Paracetamol used for? Answer:",
    "Patient: I have a mild headache. What can I take? Doctor:",

    # Category 3: General Medical Knowledge
    "Question: Which organ is responsible for pumping blood? Answer:",
    "Question: What is the main cause of anemia? Answer:",
    "Question: Is hypertension related to high blood pressure? Answer:"
]

# --- RUN BATCH TEST ---
print(f"🚀 Running 10 Test Cases on VitalLM-25M...\n")

for i, prompt in enumerate(test_prompts, 1):
    print(f"▶️ Test Case {i}:")
    # Using your existing chat function
    # Note: If it hallucinates, try chat(prompt, temp=0.2)
    chat(prompt) 
    print("\n" + "="*50 + "\n")

🚀 Running 10 Test Cases on VitalLM-25M...

▶️ Test Case 1:

User: Patient: I have a running nose, sneezing, and itchy eyes. Doctor:
VitalLM:  I have been taking a prescription for a flu shot and it is not helping. I am on a medication for the flu. I have been taking a medication for the flu and have been taking it for the past 2 weeks. I am not sure if it is a viral infection. I am on the medication for the flu. Is there anything I can do to help this?
 the mean age of the patients was 39.3  7.7 years ( range , 18 to 59 ) . 
<|endoftext|>
 the two - sided p value was considered to be statistically significant . 
 the total
------------------------------


▶️ Test Case 2:

User: Patient: I feel a burning sensation in my chest after eating spicy food. Doctor:
VitalLM:  I have a history of heartburn and a heart attack. I have been taking a lot of meds for the past 3 months. I have been taking it for about 2 weeks now. I have been taking the meds for the past 2 weeks. I am not taking any m

In [7]:
prompt = """
Q: What is the main cause of malaria?
A: Malaria is caused by a parasite transmitted by mosquitoes.

Q: What is the main cause of anemia?
A:"""

chat(prompt)


User: 
Q: What is the main cause of malaria?
A: Malaria is caused by a parasite transmitted by mosquitoes.

Q: What is the main cause of anemia?
A:
VitalLM:  The researchers found that the researchers found that the majority of the participants had a history of diabetes, hypertension, and diabetes.
 the second group was placed in the right side of the abdomen , and the other group was placed in the left side of the abdomen . 
 the most common cause of death was stroke ( 5.7% ) , followed by stroke ( 2.2% ) , stroke ( 2.2% ) , and stroke ( 1.7% ) . 
 the patient was treated with oral prednisolone and prednisone for 2 weeks . 
 the patient was treated with oral prednisolone ( 1 mg
------------------------------
